In [1]:
from hexgrid import HexCoord, HexGrid, build_obstacle_map, VelocityState, get_neighbors_at_radius, hdist
from hexstarrl import ReplayBuffer, ExperienceCollector, EnvironmentStore, HUNetTDTrainer, HUNetStateEncoder, ValueNetwork_HUNet, RLState

from hexstar import HStarProblem, HStarSearch, h_cost_travel_time_euclidean, construct_full_solution, BidHStarSearch, calculate_path_smoothness, travel_time_euclidean
from hexplot import plot_hex_grid_2

import random
from tqdm.auto import tqdm


In [2]:
import torch
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print("Device:", device)


Device: cuda


In [3]:
center = HexCoord(0,0)
start_v = VelocityState(0, None) # 0 speed, unspecified direction (allows expansion in any direction at root)
goal_v = VelocityState(0, None) # matches any direction as long as location matches. 

In [4]:
a_max = 3 # acceleration ability
a_min = 1 # deceleration ability
a_lat = 7 # maximum lateral g-force allowed by the agent

In [5]:
num_maps = 100
radius = 50
hex_size = 1
obs_densities = [0.15, 0.25] # Random range

In [6]:
n_step = 2

In [7]:
def random_start_goal(center, radius, min_distance_ratio=0.5):
    while True:
        q1 = random.randint(-radius + 1, radius - 1)
        r1 = random.randint(
            max(-radius + 1, -q1 - radius + 1),
            min(radius - 1, -q1 + radius - 1)
        )

        q2 = random.randint(-radius + 1, radius - 1)
        r2 = random.randint(
            max(-radius + 1, -q2 - radius + 1),
            min(radius - 1, -q2 + radius - 1)
        )

        start = HexCoord(center.q + q1, center.r + r1)
        goal = HexCoord(center.q + q2, center.r + r2)

        distance = (
            abs(start.q - goal.q)
            + abs(start.r - goal.r)
            + abs((start.q + start.r) - (goal.q + goal.r))
        ) // 2

        if distance >= radius * min_distance_ratio:
            return start, goal

In [8]:
# Populate the environment store

map_configs = [
    (
        "clustered_n",
        {
            "n": 100,
            "min_cluster_size": 10,
            "max_cluster_size": 150,
            # "exclude_radius": 2,
        },
    ),
    # (
    #     "random_n",
    #     {
    #         "n": 100,
    #     },
    # ),
]


replay_buffer = ReplayBuffer()
environment_store = EnvironmentStore()
experience_collector = ExperienceCollector(replay_buffer, environment_store)

from tqdm.auto import tqdm

progress = tqdm(
    range(num_maps),
    desc="Generating maps",
    unit="map",
)
for seed in progress:

    obs_density = random.uniform(*obs_densities)
    n = int(obs_density * 3 * radius * (radius + 1))
    
    map_type, map_params = random.choice(map_configs)
    #print(map_type)
    map_params['n'] = n
    start, goal = random_start_goal(
        center,
        radius,
        min_distance_ratio=0.60,
    )

    obstacles = set(
        build_obstacle_map(
            center=center,
            radius=radius,
            seed=seed,
            exclude=[start, goal],
            types=[
                (
                    map_type,
                    map_params.copy(),
                )
            ],
        ).keys()
    )

    obstacles.update(
        get_neighbors_at_radius(
            HexCoord(0, 0),
            radius,
        )
    )

    hex_grid = HexGrid(
        hex_size,
        obstacles = obstacles,
        center = center,
        radius = radius,
    )
    environment_store.add(
        seed, # environment_id
        hex_grid,
    )
    problem = HStarProblem(
        hex_grid,
        start,
        goal,
        a_max,  
        a_min,  
        start_v = start_v,
        goal_v = goal_v,
        collision_radius = 0,
        heuristic = h_cost_travel_time_euclidean,
        n_step = n_step,
        
    )
    hs = BidHStarSearch(
        problem = problem,
        bidirectional = False,
        experience_collector=experience_collector,
        environment_id = seed,
    )
    fsolution = hs.search()
    f_soln_path = construct_full_solution(fsolution)


Generating maps:   0%|          | 0/100 [00:00<?, ?map/s]

In [9]:
encoder = HUNetStateEncoder()

value_network = ValueNetwork_HUNet(
    encoder=encoder,
)

trainer = HUNetTDTrainer(
    model=value_network,
    replay_buffer=replay_buffer,
    environment_store=environment_store,
    learning_rate=1e-3,
    gamma=1.0,
)

In [10]:
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

def train_model(
    trainer,
    problem,
    search_callback=None,
    steps=10000,
    batch_size=64,
    search_every=100,
    print_every=100,
    plot=True,
):
    losses = []

    progress = tqdm(
        range(steps),
        desc="Training",
        unit="step",
    )

    for step in progress:

        # Periodically generate fresh experience.
        if (
            search_callback is not None
            and search_every
            and step % search_every == 0
        ):
            search_callback()

        metrics = trainer.train_step(
            problem,
            batch_size=batch_size,
        )

        losses.append(metrics)

        progress.set_postfix(
            loss=f"{metrics['loss']:.4f}",
            prediction=f"{metrics['mean_prediction']:.4f}",
            target=f"{metrics['mean_target']:.4f}",
        )

        if print_every and step % print_every == 0:
            tqdm.write(
                f"{step}: {metrics} | "
                f"buffer={len(trainer.replay_buffer)}"
            )

    if plot:
        loss_values = [
            metrics["loss"]
            for metrics in losses
        ]

        plt.plot(loss_values)
        plt.xlabel("Training Step")
        plt.ylabel("MSE Loss")
        plt.title("TD Training Loss")
        plt.show()

    return losses


In [11]:
def collect_experience():

    environment_id = random.choice(
        list(environment_store.environments.keys())
    )

    hex_grid = environment_store.get(
        environment_id
    )

    start, goal = random_start_goal(
        center,
        radius,
        min_distance_ratio=0.5,
    )
    
    problem = HStarProblem(
        hex_grid,
        start,
        goal,
        a_max,  
        a_min,  
        start_v = start_v,
        goal_v = goal_v,
        collision_radius = 0,
        heuristic = h_cost_travel_time_euclidean,
        n_step = n_step,   
    )
    
    hs = BidHStarSearch(
        problem=problem,
        bidirectional=False,
        experience_collector=experience_collector,
        environment_id=environment_id,
    )

    hs.search()

In [12]:
# losses = train_model(
#     trainer,
#     problem,
#     search_callback=collect_experience,
#     steps=2000,
#     batch_size=16,
#     search_every=25,
#     print_every=50,
# )

In [ ]:
losses = train_model(
    trainer,
    problem,
    search_callback=collect_experience,
    steps=1000,
    batch_size=64,
    search_every=100,
    print_every=20,
)

Training:   0%|          | 0/1000 [00:00<?, ?step/s]

0: {'loss': 0.09948597103357315, 'mean_prediction': 1.0593024492263794, 'mean_target': 1.3014509677886963} | buffer=100000


In [ ]:
for sample in replay_buffer.sample(3):

    environment = environment_store.get(
        sample.environment_id
    )
    
    prediction = value_network.predict(
        problem=problem,
        state=sample.state,
        goal=sample.goal,
        environment=environment,
    )

    euclidian_h = travel_time_euclidean(
        sample.state.location, 
        sample.goal.location, 
        sample.state.velocity.magnitude, 
        problem.a_min, 
        problem.grid.hex_size
    )
    if sample.terminal:
        target = sample.cost
    else:
        next_prediction = value_network.predict(
            problem=problem,
            state=sample.next_state,
            goal=sample.goal,
            environment=environment,
        )
    
        target = sample.cost + next_prediction
    print("------------------------------------------------")
    print("State:", sample.state)
    print("Goal:", sample.goal)
    print("Environment:", sample.environment_id)
    print("Euclidian h:", euclidian_h)
    
    print("Predicted h:", prediction)
    print("TD target:", target)
    print("Difference:", prediction - target)
print("------------------------------------------------")


In [ ]:
import torch

def plot_value_network_directions(
    value_network,
    problem,
    environment,
    goal,
    velocity: float,
    center,
    radius: int,
    hex_size: float = 1.0,
):
    """Plot predicted h across the map for the six primary hex directions."""

    directions = (
        HexCoord( 1,  0),  # E
        HexCoord( 1, -1),  # SE
        HexCoord( 0, -1),  # SW
        HexCoord(-1,  0),  # W
        HexCoord(-1,  1),  # NW
        HexCoord( 0,  1),  # NE
    )

    direction_names = (
        "E",
        "SE",
        "SW",
        "W",
        "NW",
        "NE",
    )

    value_network.eval()

    fig, axes = plt.subplots(
        2,
        3,
        figsize=(18, 12),
    )

    axes = axes.flatten()

    # Run the expensive spatial network only once.
    with torch.no_grad():
        feature_map = value_network.encode_environment(
            environment,
            goal,
        )

    obstacles = set(environment.obstacles)

    for ax, direction, direction_name in zip(
        axes,
        directions,
        direction_names,
    ):
        predictions = {}

        # Find the ordinal index corresponding to this primary direction.
        direction_index = problem.DIRECTIONS.index(
            direction
        )

        for q in range(-radius, radius + 1):
            for r in range(-radius, radius + 1):

                location = HexCoord(
                    center.q + q,
                    center.r + r,
                )

                if hdist(center, location) > radius:
                    continue

                if location in obstacles:
                    continue

                state = RLState(
                    location=location,
                    velocity=VelocityState(
                        magnitude=velocity,
                        direction=direction_index,
                    ),
                )

                prediction = value_network.predict(
                    problem=problem,
                    state=state,
                    goal=goal,
                    environment=environment,
                    feature_map=feature_map,
                )

                predictions[location] = prediction

        plot_hex_grid_2(
            obstacles=obstacles,
            start=goal.location,
            goal=goal.location,
            center=center,
            radius=radius,
            hex_size=hex_size,
            path=predictions,
            ax=ax,
            draw_axes=False,
            show_axis_legend=False,
            plt_title=(
                f"{direction_name} "
                f"{direction} | "
                f"v={velocity:.2f}"
            ),
        )

    plt.tight_layout()
    plt.show()

In [ ]:
s = replay_buffer.sample(100)[0]

goal = RLState(
    location=problem.goal,
    velocity=problem.goal_v,
)

plot_value_network_directions(
    value_network,
    problem,
    environment_store.get(s.environment_id),
    goal,
    velocity=3,
    center=center,
    radius=radius,
    hex_size=hex_size,
)

In [ ]:
s = replay_buffer.sample(100)[1]

goal = RLState(
    location=problem.goal,
    velocity=problem.goal_v,
)

plot_value_network_directions(
    value_network,
    problem,
    environment_store.get(s.environment_id),
    goal,
    velocity=3,
    center=center,
    radius=radius,
    hex_size=hex_size,
)

In [ ]:
s = replay_buffer.sample(100)[2]

goal = RLState(
    location=problem.goal,
    velocity=problem.goal_v,
)

plot_value_network_directions(
    value_network,
    problem,
    environment_store.get(s.environment_id),
    goal,
    velocity=3,
    center=center,
    radius=radius,
    hex_size=hex_size,
)

In [ ]:
s = replay_buffer.sample(100)[3]

goal = RLState(
    location=problem.goal,
    velocity=problem.goal_v,
)

plot_value_network_directions(
    value_network,
    problem,
    environment_store.get(s.environment_id),
    goal,
    velocity=3,
    center=center,
    radius=radius,
    hex_size=hex_size,
)